In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import json

from models.simple_model.model import ProtENN2_style
from models.simple_model.model_parts import CUSTOM_ALPHABET
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict

In [2]:
model_oh_easy = ProtENN2_style(cnn_dim=256,
                          kernel_size=5,
                          dilation=2,
                          in_channels=21,
                          num_pfams=131)

model_oh_hard = ProtENN2_style(cnn_dim=256,
                          kernel_size=5,
                          dilation=2,
                          in_channels=21,
                          num_pfams=248)

model_oh_easy.load_state_dict(torch.load("./models/saved_models/oh_bs_128_easy_w1-25_20e_es.pt", map_location=torch.device('cpu')))
model_oh_hard.load_state_dict(torch.load("./models/saved_models/oh_bs_128_hard_w1-25_20e_es.pt", map_location=torch.device('cpu')))

<All keys matched successfully>

In [3]:
data_easy = pd.read_parquet("./dataset/splits/easy/split_data.parquet",engine="fastparquet")

with open("./dataset/splits/easy/split.json") as json_file:
    data_split_easy = json.load(json_file)

In [4]:
data_hard = pd.read_parquet("./dataset/splits/complicated/split_data.parquet",engine="fastparquet")

with open("./dataset/splits/complicated/split.json") as json_file:
    data_split_hard = json.load(json_file)

In [5]:
len(data_easy), len(data_hard)

(25156, 24807)

In [6]:
# One hot encode the sequences
MAX_PROTEIN_LENGTH = 1000

def one_hot_encode_sequence(sequence, alphabet=CUSTOM_ALPHABET, max_length=MAX_PROTEIN_LENGTH):
    one_hot = np.zeros((max_length, len(alphabet)), dtype=np.float32)
    for i, char in enumerate(sequence):
        if i < max_length and char in alphabet:
            one_hot[i, alphabet[char]] = 1.0
    return one_hot

In [7]:
# Convert pfams to indices and pad to max length
def convert_pfams_to_indices(pfams, pfam_to_index, max_length=MAX_PROTEIN_LENGTH):
    indices = [pfam_to_index[pfam] for pfam in pfams if pfam in pfam_to_index]
    if len(indices) < max_length:
        indices += [0] * (max_length - len(indices))  # Pad with zeros
    return np.array(indices[:max_length], dtype=np.int64)

In [8]:
data_easy["sequence_oh"] = data_easy["sequence"].apply(one_hot_encode_sequence)
data_hard["sequence_oh"] = data_hard["sequence"].apply(one_hot_encode_sequence)

# Generate pfam to index mapping
pfam_to_index_easy = {pfam: idx+1 for idx, pfam in enumerate(data_easy["pfam_tensor"].explode().unique())}
pfam_to_index_hard = {pfam: idx+1 for idx, pfam in enumerate(data_hard["pfam_tensor"].explode().unique())}


data_easy["pfams_indices"] = data_easy["pfam_tensor"].apply(lambda x: convert_pfams_to_indices(x, pfam_to_index_easy))
data_hard["pfams_indices"] = data_hard["pfam_tensor"].apply(lambda x: convert_pfams_to_indices(x, pfam_to_index_hard))

In [9]:
def get_predictions(data_split, model, data):

    example_input = torch.tensor(np.stack(data.loc[data_split["test"]]["sequence_oh"].values), dtype=torch.float32)
    labels = torch.tensor(np.stack(data.loc[data_split["test"]]["pfams_indices"].values), dtype=torch.float32)

    # Create a dataset and loader
    dataset = TensorDataset(example_input)
    loader = DataLoader(dataset, batch_size=50)

    i = 0
    output = []
    for (batch_inputs,) in loader:
        suboutput = model(batch_inputs)
        suboutput = suboutput.argmax(dim=-1)
        output.append(suboutput)
        i+=1
        print(f'batch: {i}')

    tot_output = torch.cat(output).numpy()#
    
    return tot_output, labels

In [10]:
oh_preds_easy, oh_labels_easy = get_predictions(data_split_easy, model_oh_easy, data_easy)

batch: 1
batch: 2
batch: 3
batch: 4
batch: 5
batch: 6
batch: 7
batch: 8
batch: 9
batch: 10
batch: 11
batch: 12
batch: 13
batch: 14
batch: 15
batch: 16
batch: 17
batch: 18
batch: 19
batch: 20
batch: 21
batch: 22
batch: 23
batch: 24
batch: 25
batch: 26
batch: 27
batch: 28
batch: 29
batch: 30
batch: 31
batch: 32
batch: 33
batch: 34
batch: 35
batch: 36
batch: 37
batch: 38
batch: 39
batch: 40
batch: 41
batch: 42
batch: 43
batch: 44
batch: 45
batch: 46
batch: 47
batch: 48
batch: 49
batch: 50
batch: 51
batch: 52
batch: 53
batch: 54
batch: 55
batch: 56
batch: 57
batch: 58
batch: 59
batch: 60
batch: 61
batch: 62
batch: 63
batch: 64
batch: 65
batch: 66
batch: 67
batch: 68
batch: 69
batch: 70
batch: 71
batch: 72
batch: 73
batch: 74
batch: 75
batch: 76
batch: 77
batch: 78
batch: 79
batch: 80
batch: 81
batch: 82
batch: 83
batch: 84
batch: 85
batch: 86
batch: 87
batch: 88
batch: 89
batch: 90
batch: 91
batch: 92
batch: 93
batch: 94
batch: 95
batch: 96
batch: 97
batch: 98
batch: 99
batch: 100
batch: 1

In [11]:
oh_preds_hard, oh_labels_hard = get_predictions(data_split_hard, model_oh_hard, data_hard)

batch: 1
batch: 2
batch: 3
batch: 4
batch: 5
batch: 6
batch: 7
batch: 8
batch: 9
batch: 10
batch: 11
batch: 12
batch: 13
batch: 14
batch: 15
batch: 16
batch: 17
batch: 18
batch: 19
batch: 20
batch: 21
batch: 22
batch: 23
batch: 24
batch: 25
batch: 26
batch: 27
batch: 28
batch: 29
batch: 30
batch: 31
batch: 32
batch: 33
batch: 34
batch: 35
batch: 36
batch: 37
batch: 38
batch: 39
batch: 40
batch: 41
batch: 42
batch: 43
batch: 44
batch: 45
batch: 46
batch: 47
batch: 48
batch: 49
batch: 50
batch: 51
batch: 52
batch: 53
batch: 54
batch: 55
batch: 56
batch: 57
batch: 58
batch: 59
batch: 60
batch: 61
batch: 62
batch: 63
batch: 64
batch: 65
batch: 66
batch: 67
batch: 68
batch: 69
batch: 70
batch: 71
batch: 72
batch: 73
batch: 74
batch: 75
batch: 76
batch: 77
batch: 78
batch: 79
batch: 80
batch: 81
batch: 82
batch: 83
batch: 84
batch: 85
batch: 86
batch: 87
batch: 88
batch: 89
batch: 90
batch: 91
batch: 92
batch: 93
batch: 94
batch: 95
batch: 96
batch: 97
batch: 98
batch: 99
batch: 100
batch: 1

In [15]:
df_oh_easy = pd.DataFrame({"prediction": list(oh_preds_easy), "gt": list(oh_labels_easy.numpy())})
df_oh_easy

,prediction,gt
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
...,...,...
5615,"[1, 1, 130, 1, 130, 1, 130, 1, 130, 1, 130, 13...","[1.0, 1.0, 130.0, 130.0, 130.0, 130.0, 130.0, ..."
5616,"[1, 1, 130, 130, 130, 130, 130, 130, 130, 130,...","[1.0, 1.0, 130.0, 130.0, 130.0, 130.0, 130.0, ..."
5617,"[1, 1, 1, 1, 1, 130, 130, 130, 1, 130, 1, 130,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 130.0, 130.0, 1..."
5618,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 72, 1, 1,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."


In [16]:
df_oh_hard = pd.DataFrame({"prediction": list(oh_preds_hard), "gt": list(oh_labels_hard.numpy())})
df_oh_hard

,prediction,gt
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 157.0..."
2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 157.0, 157.0, 1..."
3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 226, 1, 1, 1, 226,...","[1.0, 1.0, 1.0, 1.0, 1.0, 157.0, 157.0, 157.0,..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 157.0, 157.0, 157.0,..."
...,...,...
5037,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
5038,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 234...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
5039,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
5040,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."


In [18]:
df_oh_hard["prediction"].iloc[0]

array([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
       145, 152, 145, 152, 145, 152, 145, 152, 145, 152, 145, 152, 145,
       152, 152, 152, 152, 152, 145, 145, 145, 145, 145, 145, 145, 145,
       145, 165, 145, 145, 145, 145, 145, 145, 145, 145, 145, 145, 145,
       145, 145, 145, 145, 145, 145, 145, 145, 145, 152, 145, 152, 145,
       152, 145, 152, 145, 152, 145, 152, 145, 152,  98, 152, 117, 152,
       117, 152, 145,  30,  98,  30,  98,  30, 145,  30,  98,  30, 153,
        30, 153,  30, 153,  30, 153,  30, 153,  30, 153,  30, 153,  30,
       153,  30, 153,   1, 153,   1, 153,   1, 207,   1, 207,   1, 207,
         1, 207,   1, 207,   1,  65,   1,   1,   1, 207,   1, 165,   1,
       207, 194,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,  74,   1,  74,   1,   1,   1,  74,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   0,   1,   0,   0,   

# Alternatively load data directly


In [ ]:
# OneHot Models
oh_preds_hard = np.load("ignore/hard_oh_pred.npy")
oh_labels_hard = np.load("ignore/hard_oh_labels.npy")
df_oh_hard = pd.DataFrame({"prediction": list(oh_preds_hard), "gt": list(oh_labels_hard)})

oh_preds_easy = np.load("ignore/easy_oh_pred.npy")
oh_labels_easy = np.load("ignore/easy_oh_labels.npy")
df_oh_easy = pd.DataFrame({"prediction": list(oh_preds_easy), "gt": list(oh_labels_easy)})


In [19]:
# Embedding models
emb_preds_hard = np.load("ignore/hard_emb_pred.npy")
emb_labels_hard = np.load("ignore/hard_emb_labels.npy")
df_emb_hard = pd.DataFrame({"prediction": list(emb_preds_hard), "gt": list(emb_labels_hard)})

emb_preds_easy = np.load("ignore/easy_emb_pred.npy")
emb_labels_easy = np.load("ignore/easy_emb_labels.npy")
df_emb_easy = pd.DataFrame({"prediction": list(emb_preds_easy), "gt": list(emb_labels_easy)})

In [20]:
def count_positive_negative(df):   
    pfam_tp_dict = defaultdict(int)
    pfam_fp_dict = defaultdict(int)
    pfam_fn_dict = defaultdict(int)
    n=0
    for _, row in df.iterrows():
        for idx in range(row["prediction"].shape[0]):
            pred = row["prediction"][idx]
            gt = row["gt"][idx]
            if pred == gt:
                pfam_tp_dict[gt] += 1
            else:
                pfam_fn_dict[gt] += 1
                pfam_fp_dict[pred] += 1
            n+=1
    
    
    return pfam_tp_dict, pfam_fp_dict, pfam_fn_dict, n

In [21]:
pfam_tp_dict_oh_easy, pfam_fp_dict_oh_easy, pfam_fn_dict_oh_easy, n_oh_easy = count_positive_negative(df_oh_easy)
pfam_tp_dict_oh_hard, pfam_fp_dict_oh_hard, pfam_fn_dict_oh_hard, n_oh_hard = count_positive_negative(df_oh_hard)
pfam_tp_dcit_emb_easy, pfam_fp_dict_emb_easy, pfam_fn_dict_emb_easy, n_emb_easy = count_positive_negative(df_emb_easy)
pfam_tp_dict_emb_hard, pfam_fp_dict_emb_hard, pfam_fn_dict_emb_hard, n_emb_hard = count_positive_negative(df_emb_hard)                                                                              

In [35]:
def compute_scores(pfam_tp_dict, pfam_fp_dict, pfam_fn_dict, n):    
    pfams=[]
    precision_list=[]
    recall_list=[]
    false_pos_list=[]
    accuracy_list=[]
    f1_list=[]
    balanced_accuracy_list=[]
    for pfam in pfam_fn_dict:
        tp = pfam_tp_dict[pfam]
        fp = pfam_fp_dict[pfam]
        fn = pfam_fn_dict[pfam]
        tn = n-(tp+fp+fn)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall= tp / (tp + fn)  if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        balanced_accuracy = (recall + (1 - fpr)) / 2

        precision_list.append(precision)
        recall_list.append(recall)
        false_pos_list.append(fpr)
        accuracy_list.append(accuracy)
        f1_list.append(f1)
        balanced_accuracy_list.append(balanced_accuracy)
        pfams.append(pfam)

    scores_df=pd.DataFrame({"pfams":pfams, "Precision":precision_list, "Recall":recall_list,"FPR":false_pos_list,"Accuracy": accuracy_list, "F1-score": f1_list, "Balanced Accuracy": balanced_accuracy_list})
    return scores_df

In [36]:
scores_oh_easy = compute_scores(pfam_tp_dict_oh_easy, pfam_fp_dict_oh_easy, pfam_fn_dict_oh_easy, n_oh_easy)
scores_oh_easy

,pfams,Precision,Recall,FPR,Accuracy,F1-score,Balanced Accuracy
0,1.0,0.755237,0.862375,0.119864,0.874805,0.805258,0.871256
1,6.0,0.307116,0.505188,0.000626,0.999103,0.382003,0.752281
2,7.0,0.000000,0.000000,0.000009,0.999735,0.000000,0.499995
3,8.0,0.000000,0.000000,0.000000,0.999961,0.000000,0.500000
4,3.0,0.000000,0.000000,0.000030,0.999593,0.000000,0.499985
...,...,...,...,...,...,...,...
126,126.0,0.000000,0.000000,0.000000,0.999925,0.000000,0.500000
127,127.0,0.021705,0.007453,0.001547,0.993911,0.011096,0.502953
128,128.0,0.231153,0.412313,0.000603,0.999139,0.296232,0.705855
129,129.0,0.018519,0.004854,0.000009,0.999954,0.007692,0.502422


In [37]:
scores_oh_hard = compute_scores(pfam_tp_dict_oh_hard, pfam_fp_dict_oh_hard, pfam_fn_dict_oh_hard, n_oh_hard)
scores_oh_hard

,pfams,Precision,Recall,FPR,Accuracy,F1-score,Balanced Accuracy
0,157.0,0.000000,0.000000,0.000000,0.999692,0.000000,0.500000
1,1.0,0.696480,0.849809,0.114710,0.876899,0.765542,0.867549
2,0.0,0.996761,0.998039,0.005838,0.996654,0.997399,0.996100
3,132.0,0.136971,0.078846,0.000154,0.999561,0.100081,0.539346
4,61.0,0.052356,0.000916,0.000036,0.997801,0.001801,0.500440
...,...,...,...,...,...,...,...
243,243.0,0.061937,0.049372,0.000165,0.999625,0.054945,0.524603
244,244.0,0.218579,0.198864,0.000397,0.999155,0.208256,0.599233
245,245.0,0.076125,0.086957,0.000053,0.999901,0.081181,0.543452
246,246.0,0.000000,0.000000,0.000000,0.999987,0.000000,0.500000


In [38]:
scores_emb_easy = compute_scores(pfam_tp_dcit_emb_easy, pfam_fp_dict_emb_easy, pfam_fn_dict_emb_easy, n_emb_easy)
scores_emb_easy

,pfams,Precision,Recall,FPR,Accuracy,F1-score,Balanced Accuracy
0,34.0,0.000000,0.000000,0.000000,0.994094,0.000000,0.500000
1,1.0,0.863024,0.963398,0.062620,0.944939,0.910453,0.950389
2,0.0,0.000000,0.000000,0.000000,0.419632,0.000000,0.500000
3,81.0,0.659599,0.453422,0.000343,0.998857,0.537414,0.726539
4,18.0,0.000000,0.000000,0.000000,0.992696,0.000000,0.500000
...,...,...,...,...,...,...,...
70,106.0,0.960589,0.999635,0.000052,0.999947,0.979723,0.999792
71,6.0,0.952120,0.999446,0.000106,0.999894,0.975209,0.999670
72,101.0,0.982353,0.999761,0.000052,0.999947,0.990980,0.999854
73,30.0,0.765437,0.999459,0.000131,0.999868,0.866933,0.999664


In [39]:
scores_emb_hard = compute_scores(pfam_tp_dict_emb_hard, pfam_fp_dict_emb_hard, pfam_fn_dict_emb_hard, n_emb_hard)
scores_emb_hard

,pfams,Precision,Recall,FPR,Accuracy,F1-score,Balanced Accuracy
0,1.0,0.869629,0.928803,0.043129,0.950233,0.898243,0.942837
1,5.0,0.804401,0.746032,0.000032,0.999924,0.774118,0.873000
2,0.0,0.000000,0.000000,0.000000,0.357137,0.000000,0.500000
3,7.0,0.443447,0.903108,0.000123,0.999867,0.594822,0.951492
4,9.0,0.954029,0.808877,0.000298,0.998253,0.875477,0.904289
...,...,...,...,...,...,...,...
241,226.0,0.737255,0.908213,0.000013,0.999983,0.813853,0.954100
242,229.0,0.486689,0.754497,0.000149,0.999805,0.591701,0.877174
243,225.0,0.308834,0.921256,0.000795,0.999175,0.462592,0.960230
244,243.0,0.000000,0.000000,0.000052,0.999919,0.000000,0.499974


In [40]:
scores_oh_easy.to_pickle("./dataset/scores/scores_oh_easy_all.pkl")
scores_oh_hard.to_pickle("./dataset/scores/scores_oh_hard_all.pkl")
scores_emb_easy.to_pickle("./dataset/scores/scores_emb_easy_all.pkl")
scores_emb_hard.to_pickle("./dataset/scores/scores_emb_hard_all.pkl")